---
# Layer Normalization Forward Pass 
---
In this notebook, you will implement the forward pass of a (1D) layer normalization layer in a fully vectorized way.

In [1]:
%load_ext autoreload
%autoreload 2
import tests
import train
import visual

---

## Theory

---

A 1D layer normalization layer takes an input tensor

$$
x \in \mathbb{R}^{N \times D}
$$

and normalizes each **sample** separately.
Here, $N$ is the batch size and $D$ is the dimensionality of each sample.

For one sample $x_n \in \mathbb{R}^D$, the mean $\mu_n \in \mathbb{R}$ and variance $\sigma_n^2 \in \mathbb{R}$ are computed over the $D$ dimensions of that sample:

$$
\mu_n = \frac{1}{D} \sum_{d=1}^D x_{n,d},
\qquad
\sigma^2_n = \frac{1}{D} \sum_{d=1}^D \left(x_{n,d} - \mu_n\right)^2.
$$

Then, the same mean and variance are used to normalize all dimensions of that sample:
$$\hat{x}_{n,d} = \frac{x_{n,d} - \mu_n}{\sqrt{\sigma^2_n + \varepsilon}}.$$
where $\varepsilon > 0$ is a small constant for numerical stability.  

Finally, layer normalization applies one learnable scale $\gamma_d \in \mathbb{R}$ and one learnable shift $\beta_d \in \mathbb{R}$ **per dimension**:
$$
y_{n,d} = \gamma_d \, \hat{x}_{n,d} + \beta_d,
$$
That is, the same $\gamma_d$ and $\beta_d$ are applied to all samples for dimension $d$.

The full output tensor $y$ has the same shape as the input $x$.

---
## **Task:**

Implement `init_layernorm1d` in `layernorm.py`.

Complete the task by creating the tensors needed for 1D layer normalization:

- scale tensor `gamma`, initialized with ones,
- shift tensor `beta`, initialized with zeros.

Return all two tensors with shape `(D,)`.

In [2]:
tests.test_init_layernorm1d()

✅ PASS: init_layernorm1d() is correct.


---
## **Task:**

Implement `layernorm1d_forward` in `layernorm.py`.

Complete the task by implementing the fully vectorized forward pass for 1D layer normalization.

**Hints:**
- You can use `torch.mean` and `torch.var`, which have a `dim` argument to specify the dimensions to reduce over.
- `torch.var` has a `correction` argument, which should be set to `0` for layer normalization (population variance).
- Use **broadcasting** over the batch dimension for normalization and the affine transformation.
- Layer normalization behaves the same during training and inference.

In [3]:
tests.test_layernorm1d_forward()

✅ PASS: layernorm1d_forward() is correct.


---
## Short MLP Comparison
---

Now compare two MLPs on the `Fashion-MNIST` dataset.  
Similar to MNIST, these are grayscale images with shape $(1, 28, 28)$ and 10 possible classes.  
However, they are more complex than MNIST and thus a better benchmark for our MLPs.

Look at some example images from the dataset.

In [4]:
visual.fashion_mnist_pictures(n=8).show()

100%|██████████| 26.4M/26.4M [00:02<00:00, 9.73MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 5.62MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 9.07MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.5MB/s]


The two networks we compare are:
- `BaselineMLP`, which uses fully connected layers, ReLU activations, and no normalization layers.
- `LayerNormMLP`, which uses fully connected layers, ReLU activations, and 1D layer normalization layers.

We use deeper MLPs than in previous exercises to better show the effect of layer normalization.  
We tuned the hyperparameters individually for each model to give them a good chance to perform well.

The training run is intentionally short so that it also works on CPU. If CUDA is available, it is used automatically.

In [5]:
models, histories = train.train_short_comparison(
    epochs=5,
    batch_size=128,
    max_train_samples=4000,
    max_test_samples=1000,
)

visual.show_training_comparison(histories)

Using device: cpu

Training baseline


Train loss: 0.445, Test accuracy: 80.20 %: 100%|██████████| 5/5 [00:11<00:00,  2.37s/it]



Training layer-norm


Train loss: 0.438, Test accuracy: 84.10 %: 100%|██████████| 5/5 [00:10<00:00,  2.09s/it]


Final test accuracy: baseline=80.20 %, layer-norm=84.10 %, delta=+3.90 percentage points
